# 105 --- Real-Time Streaming with FPSS

ThetaDataDx includes an FPSS (Fast Push Streaming Service) client
that delivers real-time market data over a persistent TCP connection.
Events are buffered internally; you poll with `next_event()`.

This notebook covers:

1. Connecting the FPSS client
2. Subscribing to AAPL quote updates
3. Collecting events for a time window
4. Displaying quote updates in a table
5. Subscribing to trade data
6. Showing trade flow (price, size, exchange)
7. Clean unsubscribe and shutdown

**Note:** FPSS streaming requires a market data subscription that
includes real-time access. Data is only available during market hours.

In [1]:
import time
import pandas as pd
from datetime import datetime, timedelta

from thetadatadx import Credentials, Config, ThetaDataDx

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Connect to FPSS Streaming

The `ThetaDataDx` client handles both historical and streaming.
Call `start_streaming()` to open the persistent FPSS connection.
Events arrive on a background thread and are queued internally.

In [2]:
creds = Credentials.from_file("creds.txt")
config = Config.dev()

tdx = ThetaDataDx(creds, config)
tdx.start_streaming()
print(tdx)

# Wait for the login_success event
login_event = tdx.next_event(timeout_ms=5000)
if login_event and login_event["kind"] == "login_success":
    print(f"Authenticated! Permissions: {login_event['detail']}")
else:
    print(f"Unexpected first event: {login_event}")

ThetaDataDx(historical=connected, streaming=connected)
Authenticated! Permissions: STOCK.STANDARD, OPTION.STANDARD, INDEX.FREE


## 2. Subscribe to AAPL Quotes

`subscribe_quotes()` returns a request ID. The server confirms
with a `req_response` event, then streams decoded `quote` events
with named fields (`bid`, `ask`, `bid_size`, `ask_size`, `ms_of_day`, etc.).

In [3]:
req_id = tdx.subscribe_quotes("AAPL")
print(f"Subscribe request ID: {req_id}")

# Drain the confirmation event(s)
for _ in range(5):
    ev = tdx.next_event(timeout_ms=2000)
    if ev is None:
        break
    print(f"  [{ev['kind']}] detail={ev.get('detail')} id={ev.get('id')}")
    if ev["kind"] == "req_response":
        break

Subscribe request ID: None


  [req_response] detail=Subscribed id=1


## 3. Collect Quote Events for 10 Seconds

Each `quote` event is fully decoded by the Rust core --- fields like
`bid`, `ask`, `bid_size`, `ask_size`, and `ms_of_day` are available
directly as dict keys. We collect all events within a 10-second window.

In [4]:
quote_events = []
deadline = time.monotonic() + 10  # 10 seconds

print("Collecting AAPL quotes for 10 seconds...")
while time.monotonic() < deadline:
    remaining_ms = int((deadline - time.monotonic()) * 1000)
    if remaining_ms <= 0:
        break
    ev = tdx.next_event(timeout_ms=min(remaining_ms, 1000))
    if ev is None:
        continue
    if ev["kind"] == "quote":
        quote_events.append({
            "timestamp": datetime.now(),
            "bid": ev["bid"],
            "ask": ev["ask"],
            "bid_size": ev["bid_size"],
            "ask_size": ev["ask_size"],
            "ms_of_day": ev["ms_of_day"],
            "contract_id": ev["contract_id"],
        })
    elif ev["kind"] in ("error", "server_error", "disconnected"):
        print(f"  Warning: {ev['kind']}: {ev.get('detail')}")

print(f"Collected {len(quote_events)} quote events in 10 seconds")

Collected 488 quote events in 10 seconds


## 4. Display Quote Updates in a Table

Each decoded quote event contains the full NBBO tick with named fields.
We display bid, ask, spread, and sizes for each received event.

In [5]:
if quote_events:
    df_quotes = pd.DataFrame([
        {
            "time": ev["timestamp"].strftime("%H:%M:%S.%f")[:-3],
            "bid": ev["bid"],
            "ask": ev["ask"],
            "spread": ev["ask"] - ev["bid"],
            "bid_size": ev["bid_size"],
            "ask_size": ev["ask_size"],
        }
        for ev in quote_events
    ])

    print(f"Quote event rate: {len(quote_events) / 10:.1f} events/sec")
    print(f"Mean spread: ${df_quotes['spread'].mean():.4f}")
    print()
    df_quotes.head(20)
else:
    print("No quote events received (market may be closed).")

Quote event rate: 48.8 events/sec
Mean spread: $0.0645



## 5. Subscribe to AAPL Trades

In [6]:
trade_req_id = tdx.subscribe_trades("AAPL")
print(f"Trade subscribe request ID: {trade_req_id}")

# Drain confirmation
for _ in range(5):
    ev = tdx.next_event(timeout_ms=2000)
    if ev is None:
        break
    print(f"  [{ev['kind']}] detail={ev.get('detail')} id={ev.get('id')}")
    if ev["kind"] == "req_response":
        break

Trade subscribe request ID: None
  [req_response] detail=Subscribed id=2


In [7]:
trade_events = []
other_events = []
deadline = time.monotonic() + 10

print("Collecting AAPL trades + quotes for 10 seconds...")
while time.monotonic() < deadline:
    remaining_ms = int((deadline - time.monotonic()) * 1000)
    if remaining_ms <= 0:
        break
    ev = tdx.next_event(timeout_ms=min(remaining_ms, 1000))
    if ev is None:
        continue
    if ev["kind"] == "trade":
        trade_events.append({
            "timestamp": datetime.now(),
            "price": ev["price"],
            "size": ev["size"],
            "exchange": ev["exchange"],
            "condition": ev["condition"],
            "ms_of_day": ev["ms_of_day"],
            "contract_id": ev["contract_id"],
        })
    else:
        other_events.append(ev["kind"])

print(f"Trade events:  {len(trade_events)}")
print(f"Other events:  {len(other_events)} ({', '.join(set(other_events)) if other_events else 'none'})")

Trade events:  5514
Other events:  11032 (contract_assigned, market_open, ohlcvc, quote)


## 6. Trade Flow Summary

Show a summary of the trade stream activity.

In [8]:
if trade_events:
    df_trades = pd.DataFrame([
        {
            "time": ev["timestamp"].strftime("%H:%M:%S.%f")[:-3],
            "price": ev["price"],
            "size": ev["size"],
            "exchange": ev["exchange"],
            "condition": ev["condition"],
        }
        for ev in trade_events
    ])

    print(f"Trade event rate: {len(trade_events) / 10:.1f} events/sec")
    print(f"Total volume: {df_trades['size'].sum():,} shares")
    print(f"VWAP: ${(df_trades['price'] * df_trades['size']).sum() / df_trades['size'].sum():.2f}")
    print()
    df_trades.head(20)
else:
    print("No trade events received (market may be closed).")

Trade event rate: 551.4 events/sec
Total volume: 314,746 shares
VWAP: $209.33



## 7. Unsubscribe and Shutdown

Always unsubscribe from active streams and shut down the client
to release resources and close the TCP connection cleanly.

In [9]:
# Unsubscribe from both streams
unsub_q = tdx.unsubscribe_quotes("AAPL")
unsub_t = tdx.unsubscribe_trades("AAPL")
print(f"Unsubscribe quotes req_id: {unsub_q}")
print(f"Unsubscribe trades req_id: {unsub_t}")

# Drain any remaining events
remaining = 0
while True:
    ev = tdx.next_event(timeout_ms=500)
    if ev is None:
        break
    remaining += 1
print(f"Drained {remaining} remaining events")

Unsubscribe quotes req_id: None
Unsubscribe trades req_id: None


Drained 78 remaining events


In [10]:
# Stop streaming (stops background threads, closes TCP connection)
tdx.stop_streaming()
print(tdx)  # Should show ThetaDataDx(streaming=stopped)
print("Clean shutdown complete.")

ThetaDataDx(historical=connected, streaming=none)
Clean shutdown complete.


---

**Back to start:** [101 --- Getting Started](101_getting_started.ipynb)